##Streamlining the Customer Grievance Process - Aamir Mohammed

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
df = pd.read_excel('banking_complaints_2023.xlsx')
df.head()

,Complaint ID,Date Received,Banking Product,Department,Issue ID,Complaint Description,State,ZIP,Bank Response
0,CID76118977,2023-01-01,Checking or savings account,CASA,I_3510635,on XX/XX/XX22 I opened a safe balance account ...,California,92311,Closed with monetary relief
1,CID98703933,2023-01-01,"Credit reporting, credit repair services, or o...",Credit Reports,I_3798538,There is an item from Bank of ABC on my credit...,California,91344,Closed with explanation
2,CID52036665,2023-01-01,Checking or savings account,CASA,I_3648593,On XX/XX/XX22 I found out that my account was ...,New York,10466,Closed with monetary relief
3,CID62581335,2023-01-01,Credit card or prepaid card,Credit Cards,I_6999080,I've had a credit card for years with Bank of ...,California,92127,Closed with monetary relief
4,CID65731164,2023-01-01,Checking or savings account,CASA,I_3648593,This issue has to do with the way that Bank of...,New Jersey,7946,Closed with explanation


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7011 entries, 0 to 7010
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   Complaint ID           7011 non-null   object        
 1   Date Received          7011 non-null   datetime64[ns]
 2   Banking Product        7011 non-null   object        
 3   Department             7011 non-null   object        
 4   Issue ID               7011 non-null   object        
 5   Complaint Description  7011 non-null   object        
 6   State                  6984 non-null   object        
 7   ZIP                    6981 non-null   object        
 8   Bank Response          7011 non-null   object        
dtypes: datetime64[ns](1), object(8)
memory usage: 493.1+ KB


In [ ]:
df.shape

(7011, 9)

In [ ]:
df.isnull().sum()

,0
Complaint ID,0
Date Received,0
Banking Product,0
Department,0
Issue ID,0
Complaint Description,0
State,27
ZIP,30
Bank Response,0


In [ ]:
df["Date Received"].min(), df["Date Received"].max()


(Timestamp('2023-01-01 00:00:00'), Timestamp('2023-10-21 00:00:00'))

###Reading the data

- The dataset contains about 7000 customer complaints recorded in 2023
- Each section includes the banking product, department related to the complaint, the complaint itself, and the bank's response
- The 'Complaint Description' is the main part that is used in the NLP
- The 'Department' column will then be used for classifying the complaint into specified sections

### Preprocessing the Text

- Lowercasing
- Removing punctuation
- Tokenization
- Stopword removal
- Lemmatization

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(tag):
    if tag.startswith('J'):
      return wordnet.ADJ
    elif tag.startswith('V'):
      return wordnet.VERB
    elif tag.startswith('N'):
      return wordnet.NOUN
    elif tag.startswith('R'):
      return wordnet.ADV
    else:
      return wordnet.NOUN

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


In [ ]:
from nltk.tag import pos_tag
def preprocess_text(text):
  text = text.lower()

  text = re.sub(r"[^a-z\s]", " ", text)

  words = nltk.word_tokenize(text)

  words = [word for word in words if word not in stop_words]

  pos_tags = pos_tag(words)

  lemmatized_tok = [
      lemmatizer.lemmatize(word, get_wordnet_pos(tag))
      for word, tag in pos_tags
  ]

  return " ".join(lemmatized_tok)

In [ ]:
# apply preprocessing to complaints
df["clean_text"] = df["Complaint Description"].apply(preprocess_text)
df[["Complaint Description", "clean_text"]].head()


,Complaint Description,clean_text
0,on XX/XX/XX22 I opened a safe balance account ...,xx xx xx open safe balance account online use ...
1,There is an item from Bank of ABC on my credit...,item bank abc credit report belong must remove...
2,On XX/XX/XX22 I found out that my account was ...,xx xx xx find account frozen apparent reason g...
3,I've had a credit card for years with Bank of ...,credit card year bank abc xx xx xxxx pay balan...
4,This issue has to do with the way that Bank of...,issue way bank abc account link bill pay part ...


In [ ]:
df['clean_text']


,clean_text
0,xx xx xx open safe balance account online use ...
1,item bank abc credit report belong must remove...
2,xx xx xx find account frozen apparent reason g...
3,credit card year bank abc xx xx xxxx pay balan...
4,issue way bank abc account link bill pay part ...
...,...
7006,spoke xxxx xxxx capital one xx xx xxxx explain...
7007,pnc customer service letter receive contradict...
7008,go refinance another company due xxxx xxxx tim...
7009,xxxx letter mail transunion via certified mail...


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)
tf_matrix = tf.fit_transform(df['clean_text'])
tf_matrix.shape

(7011, 5000)

In [ ]:
target = df['Department']
target.value_counts()

,count
Department,
CASA,1655
Credit Cards,1609
Loans,1111
Credit Reports,1109
Mortgage,848
Remittance,422
Others,257


###TF-IDF
- The cleaned complaint text, after its formatting was removed, was converted into numerical features using TF-IDF vectorizing
- This makes certain words have higher weight for words that appear frequently within the complaint

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(
    tf_matrix,
    target,
    test_size=0.2,
    random_state=42,
    stratify=target
)


In [ ]:
mdl = LogisticRegression(max_iter=2000)
mdl.fit(X_train, y_train)

LogisticRegression(max_iter=2000)

In [ ]:
y_pred = mdl.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print('\n', classification_report(y_test, y_pred))

Accuracy: 0.7455452601568069

                 precision    recall  f1-score   support

          CASA       0.68      0.82      0.74       331
  Credit Cards       0.74      0.74      0.74       322
Credit Reports       0.77      0.73      0.75       222
         Loans       0.75      0.76      0.76       222
      Mortgage       0.91      0.89      0.90       170
        Others       0.93      0.25      0.40        51
    Remittance       0.62      0.47      0.54        85

      accuracy                           0.75      1403
     macro avg       0.77      0.67      0.69      1403
  weighted avg       0.75      0.75      0.74      1403



###Complaint Classification Model
- Logistic Regression model was trained to predict the department that the complaint will go to based on the text of the complaint
- Model achieved ~75% accuracy on the test set, performing higher or worse depending on the department
- The results indicate that complaints contain strong enough signals to indicate which department each complaint could used to be handled with

In [ ]:
!pip install vaderSentiment

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 3.7 MB/s eta 0:00:00


In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

In [ ]:
df['VADER_sentiment'] = df['Complaint Description'].apply(
    lambda x: analyzer.polarity_scores(x)['compound']
)

df[[
    'Complaint Description',
    'VADER_sentiment'
]].head()

,Complaint Description,VADER_sentiment
0,on XX/XX/XX22 I opened a safe balance account ...,-0.9488
1,There is an item from Bank of ABC on my credit...,-0.1207
2,On XX/XX/XX22 I found out that my account was ...,-0.5267
3,I've had a credit card for years with Bank of ...,-0.8796
4,This issue has to do with the way that Bank of...,0.9357


In [ ]:
def categorize_sentiment(score):
    if score >= 0.05:
        return 'Positive'
    elif score <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

df['VADER_sentiment_category'] = df['VADER_sentiment'].apply(categorize_sentiment)
df[['clean_text', 'VADER_sentiment', 'VADER_sentiment_category']].head()

,clean_text,VADER_sentiment,VADER_sentiment_category
0,xx xx xx open safe balance account online use ...,-0.9488,Negative
1,item bank abc credit report belong must remove...,-0.1207,Negative
2,xx xx xx find account frozen apparent reason g...,-0.5267,Negative
3,credit card year bank abc xx xx xxxx pay balan...,-0.8796,Negative
4,issue way bank abc account link bill pay part ...,0.9357,Positive


In [ ]:
df["VADER_sentiment_category"].value_counts(normalize=True) * 100

,proportion
VADER_sentiment_category,
Negative,54.485808
Positive,40.992726
Neutral,4.521466


In [ ]:
df.groupby('Department')['VADER_sentiment_category'].value_counts(normalize=True) * 100

Department      VADER_sentiment_category
CASA            Negative                    66.344411
                Positive                    27.734139
                Neutral                      5.921450
Credit Cards    Negative                    49.036669
                Positive                    48.228713
                Neutral                      2.734618
Credit Reports  Positive                    53.832281
                Negative                    41.118124
                Neutral                      5.049594
Loans           Negative                    55.175518
                Positive                    39.333933
                Neutral                      5.490549
Mortgage        Negative                    50.471698
                Positive                    45.165094
                Neutral                      4.363208
Others          Negative                    59.533074
                Positive                    39.688716
                Neutral                      0.778210
Remittance      Negative                    67.061611
                Positive                    28.436019
                Neutral                      4.502370
Name: proportion, dtype: float64

###VADER Sentiment
- Applied VADER sentiment analysis on the dataset and each complaint
- The scores were mostly negative since the dataset comprised of complaints
- However some scores appeared positive after VADER, which may be the case as they may have been more positive in tone or some words, but usually this may suggest VADER struggled with formal or polite sounding complaints

In [ ]:
!pip install transformers

In [ ]:
from  transformers import pipeline

model_name = 'distilbert-base-uncased-finetuned-sst-2-english'
sent_transformer = pipeline('sentiment-analysis', model=model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Device set to use cpu


In [ ]:
sample = df['Complaint Description'].iloc[4]
sent_transformer(sample)

[{'label': 'NEGATIVE', 'score': 0.9992176294326782}]

In [ ]:
#apply model to subset of data

df_sample = df.head(500).copy()

df_sample['Transformer_sentiment'] = df_sample['Complaint Description'].apply(
    lambda x: sent_transformer(x, truncation=True)[0]['label']
)

df_sample[['Complaint Description', 'VADER_sentiment','Transformer_sentiment']].head()

,Complaint Description,VADER_sentiment,Transformer_sentiment
0,on XX/XX/XX22 I opened a safe balance account ...,-0.9488,NEGATIVE
1,There is an item from Bank of ABC on my credit...,-0.1207,NEGATIVE
2,On XX/XX/XX22 I found out that my account was ...,-0.5267,NEGATIVE
3,I've had a credit card for years with Bank of ...,-0.8796,NEGATIVE
4,This issue has to do with the way that Bank of...,0.9357,NEGATIVE


### Pretrained transformer analysis
- Applying a pretrained DistilBERT model to help further make the sentiment analysis better
- This transformer helps to interpret context as well as the tone of the words, allowing for better accuracy in classifying positivity or negativity
- In turn, made the results more accurate than before, correcting some positive labels for negative complaints

In [ ]:
pd.crosstab(df_sample['VADER_sentiment_category'], df_sample['Transformer_sentiment'])

Transformer_sentiment,NEGATIVE,POSITIVE
VADER_sentiment_category,,
Negative,303,0
Neutral,18,2
Positive,176,1


In [ ]:
df_sample['Department'].value_counts(normalize=True)*100

,proportion
Department,
CASA,37.6
Credit Cards,31.4
Credit Reports,12.6
Loans,7.2
Mortgage,6.2
Remittance,5.0


In [ ]:
df_sample.groupby('Department')['Transformer_sentiment'].value_counts(normalize=True)*100

Department      Transformer_sentiment
CASA            NEGATIVE                 100.000000
Credit Cards    NEGATIVE                  99.363057
                POSITIVE                   0.636943
Credit Reports  NEGATIVE                  98.412698
                POSITIVE                   1.587302
Loans           NEGATIVE                 100.000000
Mortgage        NEGATIVE                  96.774194
                POSITIVE                   3.225806
Remittance      NEGATIVE                 100.000000
Name: proportion, dtype: float64

## Key Findings
- The complained volume is heavily concentrated in CASA and Credit Card departments, with around ~70% of all. Improvements in these areas will have greatest impact on customers.
- The logistic regression Classifier seems to successfully predict complaint classification with about ~75% accuracy. This shows that NLP can support automated routing of complaints.
- The VADER analysis detected pretty accurately the sentiment of the complaints, but struggled with tone of some of the complaints and classified some of them as positive.
- Then after, the DistilBERT transformer outperforms VADER in detecting tone in the sentiment, showing transformer-based
- Overall, sentiment analysis conforms widespread negativity across all departments, useful for prioritization and escalation decisions.

##  Business Insights & Strategy Recommendations
1. This system will allow for the bank to continously monitor and update departments to lower negative sentiment over time.
  - Specifically making sure to make average sentiment score more positive in Credit Card and CASA departments since those are the most prevalent it seems
  - Makes sure that employees are then able to reach out to customers to make sure they are taken care of.
  -Risk Factors can be detected early with this system of sentiment analysis

2. The system could allow for better detection of stronger keywords like fraud, account closure, and other terms that will allow quicker response times by more experienced or senior staff.
  - High Priority case handling is very important to maintain customer satisfaction.
  - Helps with streamlining complaints as well and creates a system of accountability with employees as to make sure high priority cases are handled efficiently
  - Will reduce customer contact rate, potential churn rate, as well as complaints if issues are taken care of with urgency depending on the complaint score.

3. Bank will have to make sure to regularly update the model to allow for increased accuracy as well with regards to the scores.
- Updated sentiment analysis models will make sure that the bank complaint system keeps up with each update to how complaints are handled
- It will also soon allow the bank then to move from reactive complaints from customers to proactive improvements by prioritizing highly dissatisfied customers.

In [ ]:
##